# Prédiction de la quantité d'eau d'irrigation journalière

## Rapport de modélisation (régression supervisée)

**Contexte.** Ce document présente la démarche complète pour estimer la variable cible `quantite_eau_necessaire_L_jour` (litres par jour) à partir de variables agronomiques et climatiques. Il est rédigé pour un lecteur débutant en apprentissage automatique et peut servir de support de méthodologie pour un projet de fin d'études (PFE).

**Objectif pédagogique.** Pour chaque grande étape (jeu de données, analyse exploratoire, pré-traitement, modélisation, évaluation, interprétation), le texte explique le *pourquoi* et le *comment*, puis le code illustre la mise en œuvre concrète.

**Données.** Fichier `golla_dataset_100k_no_accents.csv` (100 000 observations, 13 colonnes), placé dans le même répertoire que ce notebook.

**Livrable modèle.** À la fin du notebook, le pipeline retenu (pré-traitement + XGBoost) est sauvegardé au format `joblib` sous `model/modele_xgboost_pipeline.pkl`, compatible avec l'API Flask du projet.


## Environnement Python (important)

Ce notebook utilise **XGBoost** et **scikit-learn**. Si vous voyez `ModuleNotFoundError: No module named 'xgboost'` :

1. **Sélectionner le bon noyau Jupyter** : choisissez l’interpréteur du virtualenv du projet, par exemple  
   `gollasense-api/venv/bin/python`  
   (dans VS Code : *Select Kernel* ; dans Jupyter : *Kernel > Change kernel*).

2. **Ou installer dans l’environnement actuel du notebook** (une seule fois), en exécutant dans un terminal **le même Python que le noyau** :

```bash
pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib
```

3. Depuis la racine `gollasense-api` avec le venv du projet :

```bash
source venv/bin/activate   # macOS / Linux
pip install -r requirements.txt
```


## Sommaire de la méthodologie

1. **Problème et formulation** : type de tâche (régression), variable cible, variables explicatives.
2. **Chargement et compréhension des données** : dimensions, types, valeurs manquantes, premières lignes.
3. **Analyse exploratoire (EDA)** : statistiques descriptives, distributions, variables catégorielles, liaison linéaire entre variables numériques (corrélations).
4. **Pré-traitement** : séparation apprentissage / test, encodage des catégories, normalisation des numériques, `Pipeline` scikit-learn.
5. **Modélisation** : comparaison de plusieurs algorithmes de régression (régression linéaire, forêt aléatoire, XGBoost).
6. **Évaluation** : métriques $R^2$, MAE, RMSE, lecture des résultats et limites.
7. **Interprétation** : graphiques de diagnostic (valeurs réelles vs prédites, résidus), importance des variables pour le modèle à gradients.
8. **Sauvegarde** : sérialisation du pipeline final pour déploiement.


## 1. Problème et formulation

On cherche à prédire une **quantité continue** : il s'agit d'un problème de **régression supervisée**.

- **Variable cible ($y$)** : `quantite_eau_necessaire_L_jour`, exprimée en litres par jour pour la parcelle considérée (la surface en m$^2$ est une variable d'entrée : la cible n'est pas une densité par m$^2$ forcément constante).
- **Variables explicatives ($X$)** : espèce (`nom_plante`), type de culture (`type_plante`), `type_sol`, indicateurs climatiques (`temperature`, `humidite`, `ensoleillement_h`, `vent_kmh`, `pluviometrie_mm`), `saison`, `surface_m2`, `age_plante_jours`, `etat_sante`.

**Hypothèse de travail.** Les relations entre $X$ et $y$ peuvent être non linéaires et impliquer des interactions ; des modèles linéaires et ensemblistes sont comparés pour mesurer le gain de complexité.


## 2. Environnement logiciel et imports

Les bibliothèques ci-dessous couvrent la manipulation des données (`pandas`, `numpy`), la visualisation (`matplotlib`, `seaborn`), le découpage et les métriques (`sklearn`), le modèle à gradients (`xgboost`) et la sauvegarde du modèle (`joblib`).

**Remarque.** Exécuter ce notebook depuis le dossier `gollasense-api` (chemins relatifs vers le CSV et le dossier `model/`).


In [ ]:
# Configuration d'affichage (figures nettes pour rapport)
%matplotlib inline

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 11

RANDOM_STATE = 42
DATA_PATH = "golla_dataset_100k_no_accents.csv"
MODEL_DIR = "model"
MODEL_NAME = "modele_xgboost_pipeline.pkl"

# Échantillon dédié uniquement aux graphiques EDA (accélère l'affichage sans changer la logique)
EDA_SAMPLE_SIZE = 12_000


## 3. Chargement des données et première inspection

Les commandes suivantes chargent le fichier CSV et affichent la forme du tableau (`shape`), les premières lignes (`head`) et un résumé statistique (`describe`).

**Lecture du `shape`.** Le premier nombre est le nombre d'observations (lignes), le second le nombre de variables (colonnes). Un jeu de 100 000 lignes est suffisant pour entraîner des modèles non linéaires tout en restant gérable sur une machine personnelle.


In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dimensions (lignes, colonnes) :", df.shape)
print("\nColonnes :\n", df.columns.tolist())
df.head(10)


In [ ]:
df.info()


**Interprétation de `info()`.** Les colonnes de type `object` sont des chaînes (souvent des catégories). Les `float64` et `int64` sont des variables numériques continues ou entières. La mémoire occupée dépend du nombre de lignes et du type de chaque colonne.


In [ ]:
missing = df.isnull().sum()
print("Nombre de valeurs manquantes par colonne :\n")
print(missing)
assert missing.sum() == 0, "Des valeurs manquantes sont présentes : traiter avant modélisation."


**Valeurs manquantes.** Si toutes les colonnes affichent 0, aucun imputation n'est nécessaire pour ce jeu de données. En pratique terrain, des capteurs défaillants pourraient produire des `NaN` : il faudrait alors une stratégie (imputation, suppression, modèle dédié).


In [ ]:
df.describe().T


**Lecture du tableau `describe()`.** Pour chaque variable numérique, on obtient le nombre de valeurs non nulles (`count`), la moyenne (`mean`), l'écart-type (`std`), les quartiles et le min/max.

- Une **cible** avec des valeurs toujours positives et une dispersion modérée est cohérente avec une consommation d'eau bornée inférieurement (souvent > 0) et variable selon les conditions.
- Les variables climatiques doivent rester dans des plages physiquement plausibles (température, humidité relative, etc.) : des valeurs aberrantes extrêmes pourraient être détectées par des boxplots (voir section EDA).


## 4. Analyse exploratoire des données (EDA)

L'EDA vise à décrire les données **sans** utiliser la partition test : on forme une intuition sur la distribution de $y$, sur les modalités des catégories, et sur les associations linéaires entre variables numériques.

**Note méthodologique.** Les graphiques ci-dessous utilisent un **sous-échantillon aléatoire** (`EDA_SAMPLE_SIZE`) fixé par `random_state` pour des temps d'exécution raisonnables. Les statistiques tabulaires (`describe`, corrélations sur l'échantillon) sont des **estimations** de celles de la population complète ; pour un rapport final, on peut recalculer les corrélations sur l'intégralité des données si le temps de calcul le permet.


In [ ]:
df_eda = df.sample(n=min(EDA_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df_eda["quantite_eau_necessaire_L_jour"], kde=True, ax=ax, color="steelblue")
ax.set_title("Distribution de la variable cible (échantillon EDA)")
ax.set_xlabel("quantite_eau_necessaire_L_jour (L / jour)")
ax.set_ylabel("Fréquence")
plt.tight_layout()
plt.show()


**Interprétation de l'histogramme de la cible.** La forme de la distribution indique si la cible est asymétrique, multimodale ou proche d'une cloche. Une queue vers la droite signifie quelques parcelles à besoin en eau élevé. Pour une régression, des distributions très asymétriques peuvent inciter à tester une transformation (log) ; ici on conserve la cible brute pour rester aligné avec le modèle déjà déployé dans l'application.


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_no_target = [c for c in num_cols if c != "quantite_eau_necessaire_L_jour"]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
for i, col in enumerate(num_cols_no_target[:6]):
    sns.histplot(df_eda[col], kde=True, ax=axes[i], color="seagreen")
    axes[i].set_title(col)
plt.suptitle("Distributions de quelques variables numériques (échantillon EDA)", y=1.02)
plt.tight_layout()
plt.show()


**Interprétation.** Les histogrammes permettent de repérer des variables bimodales, bornées ou avec des queues lourdes. Cela guide le choix du pré-traitement (par exemple `StandardScaler` est sensible aux outliers extrêmes ; des robust scalers existent mais ici on uniformise avec `StandardScaler` pour cohérence avec le pipeline sauvegardé).


In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
n_cat = len(cat_cols)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    vc = df_eda[col].value_counts().reindex(order).fillna(0)
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=axes[i], palette="crest", orient="h")
    axes[i].set_title(f"Effectifs — {col}")
    axes[i].set_xlabel("Nombre")
for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle("Variables catégorielles (échantillon EDA)", y=1.02)
plt.tight_layout()
plt.show()


**Interprétation des diagrammes en barres (effectifs par modalité).** Chaque barre représente le nombre d'observations par modalité. Un déséquilibre marqué entre classes peut influencer certains algorithmes ; pour une régression sur une cible continue, l'enjeu principal est surtout la **couverture** du domaine des entrées (toutes combinaisons plante/saison/sol ne sont pas forcément équiprobables en réel, mais le jeu simulé peut être plus régulier).


In [ ]:
corr = df_eda.select_dtypes(include=[np.number]).corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Matrice de corrélation (Pearson, variables numériques, échantillon EDA)")
plt.tight_layout()
plt.show()


**Interprétation de la heatmap de corrélation.** Le coefficient de corrélation de Pearson mesure une relation **linéaire** entre deux variables numériques, entre $-1$ et $+1$.

- Une corrélation forte entre une variable explicative et la cible suggère un pouvoir prédictif linéaire direct.
- Une corrélation forte **entre** deux variables explicatives indique de la multicolinéarité : les coefficients d'une régression linéaire deviennent moins stables, alors que les méthodes ensemblistes (forêt, XGBoost) gèrent souvent mieux les redondances non linéaires.

La matrice ne capture pas les effets des variables catégorielles tant qu'elles ne sont pas encodées : d'où l'intérêt du modèle global après encodage.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=df_eda,
    x="surface_m2",
    y="quantite_eau_necessaire_L_jour",
    alpha=0.25,
    s=12,
    ax=ax,
)
ax.set_title("Relation surface — besoin en eau (échantillon EDA)")
ax.set_xlabel("surface_m2")
ax.set_ylabel("quantite_eau_necessaire_L_jour")
plt.tight_layout()
plt.show()


**Nuage de points surface vs cible.** On visualise une tendance globale : les parcelles plus grandes ont souvent un besoin total en eau plus élevé, sans relation strictement linéaire (d'autres facteurs interviennent). Ce graphique illustre pourquoi la surface doit être une entrée du modèle et pourquoi la cible n'est pas une simple densité constante par m$^2$.


## 5. Pré-traitement des données

### 5.1 Séparation des entrées et de la cible

On note $X$ le bloc de variables explicatives et $y$ la variable `quantite_eau_necessaire_L_jour`.

### 5.2 Découpage apprentissage / test

La fonction `train_test_split` réserve une fraction des données (ici 20 %) pour le **test**, jamais vue pendant l'ajustement du pré-traitement et du modèle sur le **train**. Cela donne une estimation honnête de la performance en généralisation. Le paramètre `random_state` fixe le tirage pour la reproductibilité.

### 5.3 Transformations

- **Variables numériques** : `StandardScaler` soustrait la moyenne et divise par l'écart-type **calculés sur le train uniquement**. Ainsi, le jeu test n'influence pas la normalisation (évite la fuite d'information).
- **Variables catégorielles** : `OneHotEncoder` crée des variables binaires par modalité. L'option `handle_unknown='ignore'` permet, en production, de gérer une catégorie jamais vue à l'entraînement en la mettant à zéro partout.

Ces étapes sont regroupées dans un `ColumnTransformer`, puis chaînées avec le régresseur dans un `Pipeline` scikit-learn : le même objet sérialisé en `joblib` reproduit exactement les transformations à l'inférence.


In [ ]:
X = df.drop(columns=["quantite_eau_necessaire_L_jour"])
y = df["quantite_eau_necessaire_L_jour"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Colonnes catégorielles :", cat_cols)
print("Colonnes numériques   :", num_cols)
print("Taille train :", X_train.shape[0], "| Taille test :", X_test.shape[0])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)


## 6. Modélisation et comparaison d'algorithmes

Trois régresseurs sont comparés dans le **même pipeline** (même pré-traitement) :

1. **Régression linéaire** : modèle simple, interprétable, suppose une structure approximativement linéaire après encodage.
2. **Forêt aléatoire (`RandomForestRegressor`)** : agrégation d'arbres de décision entraînés sur des sous-échantillons bootstrap ; capture des non-linéarités et interactions.
3. **XGBoost** : boosting de gradients avec régularisation ; souvent très performant sur données tabulaires.

Les hyperparamètres des modèles complexes ne sont pas optimisés par grille ici (focus méthodologique) ; une section « perspectives » pourrait ajouter une recherche sur validation croisée.


In [ ]:
models = {
    "Regression_lineaire": LinearRegression(),
    "Random_Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

results = []
fitted_pipelines = {}

for name, reg in models.items():
    pipe = Pipeline([("preprocessing", preprocessor), ("regressor", reg)])
    pipe.fit(X_train, y_train)
    y_pred_test = pipe.predict(X_test)
    y_pred_train = pipe.predict(X_train)

    results.append(
        {
            "modele": name,
            "R2_train": r2_score(y_train, y_pred_train),
            "R2_test": r2_score(y_test, y_pred_test),
            "MAE_test": mean_absolute_error(y_test, y_pred_test),
            "RMSE_test": float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
        }
    )
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values("R2_test", ascending=False)
results_df


## 7. Métriques d'évaluation et lecture des résultats

### Coefficient de détermination $R^2$

$R^2 = 1 - \\frac{\\sum_i (y_i - \\hat{y}_i)^2}{\\sum_i (y_i - \\bar{y})^2}$. Il compare la qualité du modèle à celle d'un prédicteur constant égal à la moyenne de $y$. Un $R^2$ proche de 1 indique une bonne explication de la variance ; un $R^2$ négatif signifie que le modèle est moins bon que la moyenne.

**$R^2$ train vs test.** Un écart important (train très élevé, test bas) suggère du **sur-apprentissage** : le modèle mémorise le bruit du train. Un écart modéré est attendu lorsque la capacité du modèle augmente.

### Erreur absolue moyenne (MAE)

$\mathrm{MAE} = \\frac{1}{n} \\sum_i |y_i - \\hat{y}_i|$. Elle s'exprime dans **les mêmes unités** que la cible (litres par jour ici) : c'est l'erreur typique en valeur absolue, facile à communiquer à un agronome.

### Racine de l'erreur quadratique moyenne (RMSE)

$\mathrm{RMSE} = \\sqrt{\\frac{1}{n} \\sum_i (y_i - \\hat{y}_i)^2}$. Elle pénalise davantage les grandes erreurs que la MAE.

**Table triée.** Le tableau ci-dessus est trié par $R^2$ test décroissant : le meilleur modèle au sens de ce critère apparaît en premier. En pratique, on peut aussi privilégier la MAE si l'interprétation métier prime.


## 8. Graphiques de diagnostic sur le meilleur modèle (jeu test)

### 8.1 Valeurs réelles vs prédictions

Si le nuage de points suit la diagonale $y = \\hat{y}$, les prédictions sont calibrées. Des points systématiquement au-dessus ou en dessous indiquent un biais.

### 8.2 Résidus

Les résidus $e_i = y_i - \\hat{y}_i$ tracés contre $\\hat{y}_i$ permettent de vérifier l'homoscédasticité (variance constante) et l'absence de structure résiduelle. Une courbe ou un « entonnoir » suggère une modélisation incomplète.


In [ ]:
best_name = results_df.iloc[0]["modele"]
best_pipe = fitted_pipelines[best_name]
y_hat = best_pipe.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_hat, alpha=0.15, s=8, c="navy")
mx = max(y_test.max(), y_hat.max())
axes[0].plot([0, mx], [0, mx], "r--", lw=1.5, label="y = y_pred")
axes[0].set_xlabel("Valeurs réelles")
axes[0].set_ylabel("Prédictions")
axes[0].set_title(f"Réel vs prédit — {best_name} (test)")
axes[0].legend()

residuals = y_test.values - y_hat
axes[1].scatter(y_hat, residuals, alpha=0.15, s=8, c="darkred")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("Prédictions")
axes[1].set_ylabel("Résidus (réel - prédit)")
axes[1].set_title("Résidus vs prédictions (test)")

plt.tight_layout()
plt.show()

print("Modèle retenu pour les graphiques :", best_name)


**Interprétation.** Un alignement serré autour de la diagonale sur le premier graphique indique une bonne capacité prédictive globale. Sur le second, les résidus centrés sur zéro sans motif clair suggèrent que le modèle n'omet pas une tendance évidente dépendant de $\\hat{y}$ ; tout motif résiduel peut motiver une transformation de la cible ou des variables supplémentaires.


## 9. Importance des variables (XGBoost)

Après encodage, le régresseur XGBoost attribue un score d'importance à chaque feature dérivée (chaque modalité one-hot a sa propre importance). Le graphique suivant affiche les 20 plus grandes valeurs pour aider à l'explication qualitative du modèle.

**Attention.** L'importance « gain » ou « weight » dépend de l'implémentation ; ici on utilise `feature_importances_` du régresseur sklearn, cohérent avec `XGBRegressor` de xgboost.


In [ ]:
xgb_pipe = fitted_pipelines["XGBoost"]
prep = xgb_pipe.named_steps["preprocessing"]
reg = xgb_pipe.named_steps["regressor"]
feat_names = prep.get_feature_names_out()
importances = pd.Series(reg.feature_importances_, index=feat_names).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
importances.sort_values().plot(kind="barh", ax=ax, color="teal")
ax.set_title("Top 20 des importances de variables (XGBoost, après pré-traitement)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

importances.sort_values(ascending=False)


**Lecture du graphique.** Les variables en tête sont celles qui contribuent le plus aux décisions des arbres du gradient boosting (splits fréquents ou gain élevé). Cela ne prouve pas la causalité physique, mais aide à documenter le comportement du modèle pour le jury du PFE.


## 10. Pipeline final, évaluation sur le test et sauvegarde

Le modèle déployé dans l'application est le **pipeline complet** (pré-traitement + XGBoost), entraîné sur `X_train` et évalué une dernière fois sur `X_test` pour reporter les métriques. La sauvegarde avec `joblib` préserve l'intégralité du pipeline : l'API Flask n'a qu'à appeler `predict` sur un `DataFrame` ayant les mêmes noms de colonnes que `X`.

**Bonnes pratiques.** Conserver la même version majeure de `scikit-learn` et `xgboost` entre entraînement et production limite les avertissements de désérialisation.


In [ ]:
final_pipeline = Pipeline(
    [
        ("preprocessing", preprocessor),
        (
            "regressor",
            XGBRegressor(
                objective="reg:squarederror",
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

final_pipeline.fit(X_train, y_train)
y_final = final_pipeline.predict(X_test)

print("--- Métriques finales (XGBoost + pipeline, jeu test) ---")
print("R2   :", round(r2_score(y_test, y_final), 4))
print("MAE  :", round(mean_absolute_error(y_test, y_final), 4))
print("RMSE :", round(float(np.sqrt(mean_squared_error(y_test, y_final))), 4))

import os

os.makedirs(MODEL_DIR, exist_ok=True)
out_path = os.path.join(MODEL_DIR, MODEL_NAME)
joblib.dump(final_pipeline, out_path)
print("Modèle sauvegardé :", out_path)


## 11. Conclusion et limites

**Synthèse.** La démarche suit le cycle standard d'un projet de science des données : compréhension des données, EDA, pré-traitement encastré dans un pipeline, comparaison de modèles, évaluation sur un jeu tenu à part, interprétation graphique et sauvegarde pour mise en production.

**Limites possibles à discuter dans le mémoire PFE.**

- Les données peuvent être **synthétiques ou simplifiées** : la généralisation à un domaine réel dépend de la qualité du capteur et du relevé terrain.
- L'**optimisation d'hyperparamètres** (validation croisée, Optuna, etc.) n'est pas détaillée ici.
- L'**incertitude** des prédictions (intervalles de prédiction) n'est pas quantifiée ; des méthodes quantiles ou bootstrap pourraient enrichir le livrable.
- Le **déploiement** (API Flask) doit utiliser exactement les mêmes noms de colonnes et la même définition de la cible que pendant l'entraînement.

**Pistes d'extension.** Données temps réel, dérive des données (data drift), réentraînement planifié, explicabilité locale (SHAP) par prédiction.
